# KDMAge

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import math
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.KDMAge)

class KDMAge(pyagingModel):
    """Klemera-Doubal biological age with sex-specific NHANES III parameters."""

    def __init__(self):
        super().__init__()
        for sex in ["male", "female"]:
            for name in ["q", "k", "s"]:
                self.register_buffer(f"{name}_{sex}", torch.empty(0))
            self.register_buffer(f"s_ba2_{sex}", torch.empty(0))

    def preprocess(self, x):
        """Apply BioAge's log1p transform to C-reactive protein.

        BioAge fits its ``lncrp`` biomarker against ``log1p(CRP in mg/dL)``, not
        ``ln`` as phenoage does; users supply the raw measurement so the same
        column can feed clocks that log it differently.

        Notes
        -----
        CRP is clamped to ``CRP_FLOOR_MG_DL`` first, matching ``PhenoAge``. The
        floor equals the registered lower bound, so any value it moves is already
        out of range and has already been warned about.
        """
        index = self.features.index("c_reactive_protei

In [3]:
model = pya.models.KDMAge()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "kdmage"
model.metadata["data_type"] = "clinical biomarkers"  # Paper: blood chemistry and organ function test data
model.metadata["species"] = "Homo sapiens"  # Paper: Homo sapiens
model.metadata["year"] = 2021
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Kwon, Dayoon, and Daniel W. Belsky. \"A toolkit for quantification of biological age from blood chemistry and organ function test data: BioAge.\" GeroScience 43.6 (2021): 2795-2808."
model.metadata["doi"] = "https://doi.org/10.1007/s11357-021-00480-5"
model.metadata["notes"] = "Klemera-Doubal biological age, trained sex-specifically on NHANES III adults aged 30-75 who were not pregnant, using the BioAge package defaults. Biomarker parameters were fit on SI-unit variants so they are natively in pyaging's unit convention, and C-reactive protein is supplied raw in mg/dL and log1p-transformed inside the clock. Sex is coded female = 1 and male = 0; a dataset with no female column scores every sample with the male parameters."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["blood"]  # Paper: blood chemistry
model.metadata["predicts"] = ["biological age"]  # Paper: Klemera-Doubal biological age
model.metadata["training_target"] = ["chronological age"]  # Paper: chronological age
model.metadata["unit"] = ["years"]  # Paper: years
model.metadata["model_type"] = "Klemera–Doubal composite"  # Paper: Klemera-Doubal biological age
model.metadata["platform"] = ["clinical laboratory assays"]  # Paper: blood chemistry and organ function test data
model.metadata["population"] = "adults"  # Paper: age >= 30 & age <= 75 & pregnant == 0
model.metadata["journal"] = "GeroScience"
model.metadata["last_author"] = "Daniel W. Belsky"
model.metadata["n_features"] = 11
model.metadata["citations"] = 332
model.metadata["citations_date"] = "2026-08-20"

## Download clock dependencies

In [5]:
with open("../bioage_params/kdmage.json") as handle:
    params = json.load(handle)

params["features"]

['forced_expiratory_volume',
 'systolic_blood_pressure',
 'total_cholesterol',
 'hemoglobin_a1c',
 'albumin',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'blood_urea_nitrogen',
 'age',
 'female']

## Load features

In [6]:
model.features = params["features"]
model.features

['forced_expiratory_volume',
 'systolic_blood_pressure',
 'total_cholesterol',
 'hemoglobin_a1c',
 'albumin',
 'creatinine',
 'c_reactive_protein',
 'alkaline_phosphatase',
 'blood_urea_nitrogen',
 'age',
 'female']

#### Normal feature ranges

In [ ]:
feature_ranges = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
model.feature_units = [record["unit"] for record in feature_ranges]
pd.DataFrame.from_records(feature_ranges)

## Load weights into base model

In [8]:
model.base_model = torch.nn.Identity()

for sex in ["male", "female"]:
    fit = params[sex]
    order = [fit["biomarkers"].index(name) for name in model.features[:-2]]
    for key in ["q", "k", "s"]:
        setattr(model, f"{key}_{sex}", torch.tensor([fit[key][index] for index in order], dtype=torch.float64))
    setattr(model, f"s_ba2_{sex}", torch.tensor(fit["s_ba2"], dtype=torch.float64))

    # The reindex above is a no-op whenever the two orders already agree, which is exactly
    # when a mangled copy of it would go unnoticed. Check it mapped what it claims.
    for position, name in enumerate(model.features[:-2]):
        source = fit["biomarkers"].index(name)
        for key in ["q", "k", "s"]:
            assert getattr(model, f"{key}_{sex}")[position].item() == fit[key][source], (sex, key, name)

model.q_female

tensor([ 3.9277, 85.5114,  3.7846,  4.4498, 41.5707, 51.6685,  0.3033, 54.9584,
         2.2111], dtype=torch.float64)

## Load reference values

In [9]:
crp = model.features.index("c_reactive_protein")
reference = [
    (male + female) / 2
    for male, female in zip(model.q_male.tolist(), model.q_female.tolist())
]
reference[crp] = math.expm1(reference[crp])  # stored raw; preprocess applies log1p

model.reference_values = reference + [52.5, 0.0]  # age: midpoint of the 30-75 window; female: male

assert len(model.reference_values) == len(model.features)
assert math.isclose(
    math.log1p(model.reference_values[crp]),
    (model.q_male[crp].item() + model.q_female[crp].item()) / 2,
)
model.reference_values

[4.617090078192,
 93.30051647306101,
 4.3532117996715005,
 4.590652017098501,
 43.670469247465505,
 60.691406723117,
 0.2575083076346611,
 65.497718435976,
 2.909765132493,
 52.5,
 0.0]

## Load preprocess and postprocess objects

In [10]:
model.preprocess_name = "log1p_crp"
model.preprocess_dependencies = None

In [11]:
model.postprocess_name = "klemera_doubal"
model.postprocess_dependencies = None

## Check all clock parameters

In [12]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Kwon, Dayoon, and Daniel W. Belsky. "A toolkit for '
             'quantification of biological age from blood chemistry and organ '
             'function test data: BioAge." GeroScience 43.6 (2021): 2795-2808.',
 'citations': 332,
 'citations_date': '2026-08-20',
 'clock_name': 'kdmage',
 'data_type': 'clinical biomarkers',
 'doi': 'https://doi.org/10.1007/s11357-021-00480-5',
 'journal': 'GeroScience',
 'last_author': 'Daniel W. Belsky',
 'model_type': 'Klemera–Doubal composite',
 'n_features': 11,
 'notes': 'Klemera-Doubal biological age, trained sex-specifically on NHANES '
          'III adults aged 30-75 who were not pregnant, using the BioAge '
          'package defaults. Biomarker parameters were fit on SI-unit variants '
          "so they are natively in pyaging's unit convention, and C-reactive "
  

## Basic test

In [13]:
records = pya.utils.resolve_feature_ranges(model.features, model.metadata["data_type"])
midpoints = torch.tensor(
    [[(record["low"] + record["high"]) / 2 for record in records]], dtype=torch.float64
)
model.eval()
model.to(torch.float64)
pred = model(midpoints)
pred

tensor([[419.3437]], dtype=torch.float64)

#### Parity with BioAge

In [14]:
with open("../bioage_params/reference_predictions.json") as handle:
    reference = json.load(handle)

matrix = torch.tensor(
    [[row[name] for name in model.features] for row in reference["rows"]], dtype=torch.float64
)
with torch.inference_mode():
    predicted = model(matrix).squeeze(-1)

expected = torch.tensor(reference["expected"]["kdmage"], dtype=torch.float64)
print("max absolute difference:", (predicted - expected).abs().max().item())

max absolute difference: 6.863842827442568e-12


## Save torch model

In [15]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [16]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)